# GreekKV sweep — Kaggle runner

Runs the GreekKV T17+T20 sweep on a Kaggle session. The notebook is a **thin wrapper** — all logic lives in `scripts/run_sweep.py` and `src/greekkv/eval/perplexity.py`. The sweep YAML drives the grid; `--config` is the only knob you'd typically turn.

Target hardware: **Kaggle T4 (16 GB)**. To use the full 7B Greek models (Meltemi / Krikri) you'll want `dtype=int4` with `bitsandbytes` — update the sweep YAML's model name and the `LoadSpec` call inside the runner accordingly (`docs/vram_table.md` lists the budget table).

## 1 — Install dependencies

In [ ]:
# Kaggle images ship torch + transformers; we only need bitsandbytes for
# the int4/int8 weight loading. A fresh clone is the simplest way to
# get the exact pinned deps from pyproject.toml.
!git clone https://github.com/georgebon5/greekkv.git --depth 1 || true
%cd greekkv
!pip install -q -e .
# OPTIONAL: install bitsandbytes for 4-bit / 8-bit model loading.
# !pip install -q bitsandbytes

## 2 — Run the T21 sweep (resumable)

The sweep YAML lists every `(scheme, n_ctx)` cell. If the output JSONL already has rows for some of them (e.g. from a previous run), they are skipped by `config_hash` — only the missing cells are computed.

In [ ]:
!python scripts/run_sweep.py --config configs/sweep_perplexity_qwen_0_5b.yaml

## 3 — Inspect the output

The JSONL schema follows `greekkv.utils.ResultRecord` (T03), so the T22 report can regenerate tables + figures from this file without a re-run.

In [ ]:
import json

rows = []
with open("results/perplexity/qwen2_5_0_5b.jsonl") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))
print(f"{len(rows)} rows")
for r in rows[:3]:
    print(r["scheme"], r["metrics"]["n_ctx"], r["metrics"]["perplexity"])

## Tips

- Killed sessions re-run the next cell for free: the JSONL is persistent and the runner skips any already-scored `(scheme, n_ctx)` cell.
- To extend the sweep (e.g. add `n_ctx=2048`), edit the YAML and re-run — only the new cells are computed.
- To compare a different model (Meltemi / Krikri), copy the YAML, set a new `output:` path, and run.